In [1]:
%load_ext autoreload
%autoreload 2

**Author:** Salvador Navas  
**Date:** 2025-06-27

In [2]:
from pyhydra.climate.time_series import (
    # extraction
    extract_block_maxima,
    extract_pot,
    threshold_stability_plot,
    # GEV fitting
    fit_gev,
    fit_gev_map,
    fit_gev_fisher,
    fit_gev_mcmc,
    # GPD fitting
    fit_gpd,
    # return levels
    return_level_gev,
    return_level_gpd,
    return_levels,
    return_level_ci,
    # diagnostics
    plot_return_levels,
    plot_diagnostic,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import genextreme

# Extreme Value Analysis

Extreme value analysis (EVA) estimates the probability of rare hydrological events — floods, droughts, extreme rainfall — that exceed observed records. It is the statistical backbone of flood-frequency analysis, design discharge estimation, and climate risk assessment.

---

## Two complementary frameworks

| Method | Distribution | Data | Record length | Best for |
|--------|-------------|------|--------------|----------|
| **Block Maxima (BM)** | GEV (Generalised Extreme Value) | Annual or seasonal maxima | ≥ 25 years | Long, complete records with clear seasonal structure |
| **Peaks Over Threshold (POT)** | GPD (Generalised Pareto) | Independent peaks above a threshold | ≥ 10–15 years | Short records; maximises use of available data |

---

## The GEV distribution — three families in one

The GEV unifies three extreme-value types via the **shape parameter ξ (xi)**:

| ξ | Family | Tail behaviour | Typical application |
|---|--------|---------------|---------------------|
| **ξ > 0** | Fréchet | Heavy tail — no upper bound | Flood discharge, precipitation maxima |
| **ξ = 0** | Gumbel | Light exponential tail | Wind speed, temperature maxima |
| **ξ < 0** | Weibull | Bounded upper tail | Sea level, bounded physical quantities |

The **return level** $z_T$ (value exceeded once every $T$ years on average) is:
$$z_T = \mu + \frac{\sigma}{\xi}\left[(-\ln(1-1/T))^{-\xi} - 1\right] \quad (\xi \neq 0)$$
$$z_T = \mu - \sigma \ln(-\ln(1-1/T)) \quad (\xi = 0)$$

---

## Fitting strategy

pyhydra applies a robust fitting cascade:
1. **L-moments** (lmoments3) — warm-start for MLE, resistant to outliers
2. **MLE** — maximum likelihood with bounded ξ ∈ [−0.5, 0.8] to avoid physically impossible estimates
3. **MAP** (Bayesian maximum a-posteriori) — useful for very short records (< 20 yr); prior on ξ prevents unrealistic heavy tails
4. **Full MCMC** (PyMC) — complete posterior distribution; quantifies parameter uncertainty explicitly

---

## Installation
```bash
pip install scipy lmoments3          # MLE + L-moments (core)
pip install pymc                     # full MCMC (optional, Bayesian section)
```

---
## Synthetic dataset

60 years of synthetic daily discharge to demonstrate all fitting methods. The series has:
- **Seasonal base flow**: smooth sinusoidal curve (80–180 m³/s)
- **Flood events**: Poisson-distributed arrivals (~4/year), log-normal peak magnitudes (heavy right tail, ξ ≈ 0.15 expected from the GEV fit)
- **Recession limbs**: exponential decay with 10-day time constant

Using synthetic data ensures the *true* distribution parameters are known, making it possible to evaluate estimator accuracy. In production, replace this series with observed daily discharge or sub-daily precipitation.

In [3]:
rng = np.random.default_rng(42)
dates = pd.date_range("1960-01-01", "2019-12-31", freq="D")
n = len(dates)

# Seasonal base flow: smooth sinusoid 80–180 m³/s
doy = np.array(dates.dayofyear, float)
base = 130 + 50 * np.sin(2 * np.pi * (doy / 365 - 0.25))

# Flood events: Poisson arrivals at mean rate 4/year.
# Peaks: log-normal with sigma_log=0.80, giving annual maxima clearly in the
# Fréchet domain (ξ > 0) for the GEV fit.
# lognormal(6.0, 0.80): median ≈ 403 m³/s, mean ≈ 556 m³/s.
flood_signal = np.zeros(n)
t = 0
while t < n:
    t += int(rng.exponential(365.0 / 4))      # mean inter-arrival ≈ 91 days
    if t >= n:
        break
    peak   = rng.lognormal(mean=6.0, sigma=0.80)   # log-normal peak (m³/s)
    rise_d = int(rng.integers(1, 4))                # 1–3 day rising limb
    k      = rng.uniform(0.12, 0.28)               # exponential recession coeff (day⁻¹)
    rise   = np.linspace(0, peak, rise_d + 1)[1:]
    fall   = peak * np.exp(-k * np.arange(1, 120))
    fall   = fall[fall > 5.0]
    seg    = np.concatenate([rise, fall])
    end    = min(t + len(seg), n)
    flood_signal[t:end] += seg[:end - t]

noise = rng.exponential(15, n)
Q = base + flood_signal + noise
Q = np.clip(Q, 10, None)
discharge = pd.Series(Q, index=dates, name="Q (m³/s)")

print(f"Period : {dates[0].date()} → {dates[-1].date()}  ({len(dates)} days)")
print(f"Mean   : {discharge.mean():.1f} m³/s")
print(f"Std    : {discharge.std():.1f} m³/s")
print(f"Max    : {discharge.max():.1f} m³/s")

Period : 1960-01-01 → 2019-12-31  (21915 days)
Mean   : 179.8 m³/s
Std    : 142.3 m³/s
Max    : 4248.6 m³/s


In [4]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(discharge.index, discharge.values, lw=0.4, color="steelblue", alpha=0.8)
ax.set_ylabel("Discharge (m³/s)")
ax.set_title("Synthetic daily discharge — 60 years", fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 1. Block Maxima — GEV

### What is the block maxima method?

Extract **one maximum per block** (typically one year) and fit a GEV distribution. The Extreme Value Theorem guarantees that block maxima from any continuous distribution converge to a GEV as block size grows — this is the theoretical justification.

### When to use block maxima vs POT
- ✅ Record ≥ 25 years, well-defined annual cycle
- ✅ Design standards require annual exceedance probabilities (AEP)
- ❌ Short records (< 15 yr) — only one value per year wastes data
- ❌ Series with multiple distinct flood seasons — consider seasonal POT instead

### The annual maximum series (AMS)
The AMS is the sequence {max(year 1), max(year 2), …}. By construction, values are **independent** (one per year), satisfying the i.i.d. assumption for GEV fitting.

In [5]:
annual_max = extract_block_maxima(discharge, freq="YE")
print(f"Annual maxima: {len(annual_max)} values")
print(f"Range: {annual_max.min():.0f} – {annual_max.max():.0f} m³/s")

fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(annual_max.index.year, annual_max.values, color="steelblue", alpha=0.7)
ax.set_xlabel("Year")
ax.set_ylabel("Annual maximum (m³/s)")
ax.set_title("Annual maxima series", fontsize=11)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

Annual maxima: 60 values
Range: 251 – 4249 m³/s


In [6]:
# Fit GEV — both MLE and L-moments
params_gev = fit_gev(annual_max.values, method="both")
print("GEV MLE:")
if "error" not in params_gev["mle"]:
    p = params_gev["mle"]
    print(f"  mu = {p['mu']:.1f}  sigma = {p['sigma']:.1f}  xi = {p['xi']:.4f}")

print("\nGEV L-moments:")
try:
    p = params_gev["lmom"]
    print(f"  mu = {p['mu']:.1f}  sigma = {p['sigma']:.1f}  xi = {p['xi']:.4f}")
except Exception:
    print("  (lmoments3 not installed — pip install lmoments3)")

params_best = params_gev.get("mle") or params_gev.get("lmom")

GEV MLE:
  mu = 864.4  sigma = 445.3  xi = 0.1480

GEV L-moments:
  mu = 859.9  sigma = 449.0  xi = 0.1486


### Interpreting the fitted GEV parameters

| Parameter | Symbol | Meaning | Typical range |
|-----------|--------|---------|--------------|
| **Location** | μ (mu) | Approximate median of annual maxima (≈ mode for ξ small) | Same units as data |
| **Scale** | σ (sigma) | Spread of the annual maxima distribution; larger σ → wider return level curve | > 0 |
| **Shape** | ξ (xi) | Tail heaviness. **ξ > 0** → unbounded Fréchet; **ξ = 0** → Gumbel; **ξ < 0** → Weibull | Typically −0.5 to +0.5 |

**L-moments vs MLE:**
- L-moments are **more robust** to outliers and almost always produce a finite estimate (they can fail only if there are ties or very small samples)
- MLE is **more efficient** (lower variance) for large samples and is needed for likelihood-ratio tests
- Use L-moments to initialise MLE (already done internally by `fit_gev`)

In [7]:
# Return levels table
T_values = [2, 5, 10, 25, 50, 100, 200, 500, 1000]
rl_gev = return_levels(params_best, T_values, dist="gev")
print("GEV return levels (m³/s):")
print(pd.DataFrame({"T (years)": T_values, "Return level (m³/s)": rl_gev.values.round(0)}).to_string(index=False))

GEV return levels (m³/s):
 T (years)  Return level (m³/s)
         2               1032.0
         5               1612.0
        10               2053.0
        25               2686.0
        50               3216.0
       100               3799.0
       200               4444.0
       500               5403.0
      1000               6219.0


In [8]:
# Return level plot with 95% bootstrap CI
fig, ax = plt.subplots(figsize=(9, 5))
plot_return_levels(annual_max.values, params_best, dist="gev",
                   n_bootstrap=100, ax=ax)
plt.tight_layout()
plt.show()

In [9]:
# Full 2×2 diagnostic panel
fig = plot_diagnostic(annual_max.values, params_best, dist="gev")
plt.show()

---
## 2. Bootstrap confidence intervals

`return_level_ci` gives (point, lower, upper) for any return period
without assuming a parametric sampling distribution.

In [10]:
print(f"{'T':>6}  {'Point':>8}  {'Lower (95%)':>11}  {'Upper (95%)':>11}")
for T in [10, 50, 100]:
    pt, lo, hi = return_level_ci(
        annual_max.values, T=T, dist="gev",
        method="mle", n_bootstrap=100, ci=0.95
    )
    print(f"{T:>6}  {pt:>8.0f}  {lo:>11.0f}  {hi:>11.0f}")

     T     Point  Lower (95%)  Upper (95%)
    10      2053         1716         2504
    50      3216         2362         4675
   100      3799         2584         5883


---
## 3. Peaks Over Threshold — GPD

### Why POT?
Block maxima use only 1 value per year. POT uses **all independent peaks above a threshold**, typically 3–10 times more events from the same record. More events → better parameter estimates → tighter confidence intervals.

### 3.1 Threshold selection — the fundamental challenge

Too low a threshold: GPD approximation breaks down (asymptotic assumption not met); fit is biased.  
Too high a threshold: Very few peaks; high variance. 

**Two diagnostic plots guide the choice:**

| Plot | What to look for |
|------|-----------------|
| **Mean excess plot** (left) | Should be approximately **linear** above the threshold. Non-linearity below indicates GPD approximation not yet valid. |
| **Shape stability plot** (right) | Estimated GPD shape ξ should be **approximately constant** (flat) over a range of thresholds. A stable plateau marks the valid range. |

**Rule of thumb:** Choose the lowest threshold where both plots stabilise, keeping at least 20–30 exceedances per year of record.

In [11]:
stab = threshold_stability_plot(discharge, min_peaks=15)
stab.head(10)

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(stab.threshold, stab.mean_excess, "b-o", ms=3)
axes[0].set_xlabel("Threshold (m³/s)")
axes[0].set_ylabel("Mean excess")
axes[0].set_title("Mean excess plot", fontsize=11)
axes[0].grid(alpha=0.3)

axes[1].plot(stab.threshold, stab.gpd_shape, "r-o", ms=3, label="Shape (xi)")
axes[1].axhline(0, ls="--", color="k", lw=0.8)
axes[1].set_xlabel("Threshold (m³/s)")
axes[1].set_ylabel("GPD shape parameter")
axes[1].set_title("Shape stability", fontsize=11)
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle("Threshold stability plots", fontsize=12)
plt.tight_layout()
plt.show()

# Choose a threshold where shape becomes stable
threshold = float(stab[stab.n_exceed >= 50].threshold.iloc[-1])
print(f"Selected threshold: {threshold:.0f} m³/s  "
      f"({stab[stab.threshold == threshold].n_exceed.values[0]:.0f} exceedances)")

Selected threshold: 358 m³/s  (1096 exceedances)


### 3.2 GPD fitting

The GPD is fitted only to exceedances **above the threshold** (y = peak − threshold). Two parameters:
- **Scale** σ_u: spread of exceedances above u (depends on threshold; not directly comparable across thresholds)
- **Shape** ξ: same tail parameter as GEV — ideally consistent with the GEV fit

The **Poisson rate** λ (peaks/year) converts exceedance probabilities to return periods:
$$T = \frac{1}{\lambda \cdot P(X > z \mid X > u)} = \frac{1}{\lambda \cdot (1 + \xi(z-u)/\sigma_u)^{-1/\xi}}$$

> **Consistency check:** The GEV shape ξ from block maxima and GPD shape ξ from POT should agree within sampling uncertainty. Large disagreement indicates model misspecification or threshold issues.

In [13]:
params_gpd = fit_gpd(discharge, threshold=threshold, method="mle", min_gap=7)
print("GPD parameters:")
print(f"  scale        = {params_gpd['scale']:.2f}")
print(f"  shape (xi)   = {params_gpd['shape']:.4f}")
print(f"  threshold    = {params_gpd['threshold']:.0f} m³/s")
print(f"  n exceedances= {params_gpd['n_exceed']}")
print(f"  lambda rate  = {params_gpd['lambda_rate']:.2f} peaks/year")

GPD parameters:
  scale        = 403.39
  shape (xi)   = 0.1198
  threshold    = 358 m³/s
  n exceedances= 181
  lambda rate  = 3.02 peaks/year


In [14]:
# Return levels table
rl_gpd = return_levels(params_gpd, T_values, dist="gpd")
print("GPD return levels (m³/s):")
print(pd.DataFrame({"T (years)": T_values, "Return level (m³/s)": rl_gpd.values.round(0)}).to_string(index=False))

GPD return levels (m³/s):
 T (years)  Return level (m³/s)
         2               1167.0
         5               1651.0
        10               2055.0
        25               2642.0
        50               3132.0
       100               3663.0
       200               4241.0
       500               5082.0
      1000               5782.0


In [15]:
# Return level plot
fig, ax = plt.subplots(figsize=(9, 5))
plot_return_levels(discharge, params_gpd, dist="gpd", n_bootstrap=100, ax=ax)
plt.tight_layout()
plt.show()

---
## 4. GEV vs GPD comparison

Both methods should give consistent return levels — if they disagree substantially, investigate:
- Too few annual maxima (< 25 yr) → GEV estimates unreliable
- Threshold too low → GPD fit biased upward
- Non-homogeneous series (trend, mixed populations) → neither method is fully valid

**Preferred method in practice:**
- **Short record (< 20 yr):** POT + GPD (more events, better estimates)
- **Long record (≥ 30 yr), design codes require AEP:** Block maxima + GEV (simpler, widely accepted)
- **Both available:** Run both and use as consistency cross-check; weight toward GPD if records are short

In [16]:
# Smooth return level curves — 200 points on a log scale.
# Start at T = 1.5 yr: at T = 1.0 exactly, the GEV lower-bound singularity
# gives mu − σ/ξ ≈ −2142 m³/s, which is unphysical.
T_fine = np.logspace(np.log10(1.5), np.log10(500), 200)
rl_gev_fine = [return_level_gev(params_best, T) for T in T_fine]
rl_gpd_fine = [return_level_gpd(params_gpd, T) for T in T_fine]

# Empirical — annual maxima (Gringorten positions, T always ≥ 1 yr)
arr_sorted = np.sort(annual_max.values)
n_am = len(arr_sorted)
prob_am = (np.arange(1, n_am + 1) - 0.44) / (n_am + 0.12)
T_emp_am = 1.0 / (1.0 - prob_am)

# Empirical — POT peaks: T = 1/(λ(1−F̂)).
# With λ = 3.02/yr, 121 of 181 peaks fall at T < 1 yr (sub-annual exceedances).
# Only the top ~60 peaks reach T ≥ 1 yr — these are the comparable range.
pot_peaks = extract_pot(discharge, threshold=params_gpd["threshold"], min_gap=7)
pot_sorted = np.sort(pot_peaks.values)
n_pot = len(pot_sorted)
prob_pot = (np.arange(1, n_pot + 1) - 0.44) / (n_pot + 0.12)
T_emp_pot = 1.0 / (params_gpd["lambda_rate"] * (1.0 - prob_pot))
mask_annual = T_emp_pot >= 1.0

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(T_fine, rl_gev_fine, "b-", lw=2, label="GEV (block maxima)")
ax.semilogx(T_fine, rl_gpd_fine, "r-", lw=2, label="GPD (POT)")
ax.semilogx(T_emp_am, arr_sorted, "bo", ms=5, alpha=0.7, label="Empirical (annual max)")
ax.semilogx(T_emp_pot[mask_annual], pot_sorted[mask_annual],
            "rs", ms=4, alpha=0.55, label="Empirical (POT, T ≥ 1 yr)")

ax.set_xlim([1, 500])
ax.set_ylim(bottom=0)
ax.set_xlabel("Return period (years)")
ax.set_ylabel("Discharge (m³/s)")
ax.set_title("Return level comparison — GEV vs GPD", fontsize=11)
ax.legend()
ax.grid(which="both", alpha=0.3)
plt.tight_layout()
plt.show()

# Numerical comparison at discrete return periods
T_comp = [2, 5, 10, 25, 50, 100, 200, 500]
rl_gev_vals = [return_level_gev(params_best, T) for T in T_comp]
rl_gpd_vals = [return_level_gpd(params_gpd, T) for T in T_comp]
comparison = pd.DataFrame({
    "T (years)": T_comp,
    "GEV (m³/s)": np.round(rl_gev_vals, 0),
    "GPD (m³/s)": np.round(rl_gpd_vals, 0),
    "Diff (%)": np.round(
        100 * (np.array(rl_gpd_vals) - np.array(rl_gev_vals)) / np.array(rl_gev_vals), 1
    ),
})
print(comparison.to_string(index=False))

 T (years)  GEV (m³/s)  GPD (m³/s)  Diff (%)
         2      1032.0      1167.0      13.1
         5      1612.0      1651.0       2.4
        10      2053.0      2055.0       0.1
        25      2686.0      2642.0      -1.6
        50      3216.0      3132.0      -2.6
       100      3799.0      3663.0      -3.6
       200      4444.0      4241.0      -4.6
       500      5403.0      5082.0      -5.9


---
## 5. Bayesian GEV

Classical MLE treats parameters as fixed unknowns. Bayesian methods treat them as **random variables with a prior distribution** and update the prior with data to obtain a posterior distribution. This is especially useful when:
- Records are short (< 20 years) — prior regularises unrealistic MLE estimates
- Full uncertainty quantification is needed — posterior gives a probability distribution over return levels, not just a point estimate

### 5.1 MAP estimation (fast, no MCMC)

MAP (Maximum A Posteriori) finds the **mode of the posterior** — equivalent to a penalised MLE where the penalty is minus the log-prior. Much faster than MCMC, captures prior regularisation.

**Priors used in pyhydra MAP:**
- μ ~ Normal(ȳ, 3σ_y) — broad prior around sample mean
- σ ~ LogNormal(log(σ_y), 1) — positive, centered on sample std
- ξ ~ Normal(0, 0.3) — pulls shape toward Gumbel (ξ=0); prevents extreme tails on short records

> **When to use MAP:** Short records (< 20 yr), when full MCMC is too slow, or as a diagnostic sanity check.

In [17]:
params_map = fit_gev_map(annual_max.values)
print("Bayesian MAP:")
print(f"  mu    = {params_map['mu']:.1f}")
print(f"  sigma = {params_map['sigma']:.1f}")
print(f"  xi    = {params_map['xi']:.4f}")
print(f"  log-posterior = {params_map['map_logpost']:.2f}")

print("\nComparison MLE vs MAP:")
print(f"  {'':10} {'MLE':>10} {'MAP':>10}")
for key in ['mu', 'sigma', 'xi']:
    mle_val = params_best.get(key, float('nan'))
    map_val = params_map[key]
    print(f"  {key:10} {mle_val:>10.2f} {map_val:>10.2f}")

Bayesian MAP:
  mu    = 875.0
  sigma = 457.0
  xi    = 0.1288
  log-posterior = -474.34

Comparison MLE vs MAP:
                    MLE        MAP
  mu             864.37     874.98
  sigma          445.27     456.96
  xi               0.15       0.13


In [18]:
# Short record demonstration — MAP vs MLE comparison
short_record = annual_max.values[:15]   # only 15 years

try:
    params_mle_short = fit_gev(short_record, method="mle")
except RuntimeError:
    params_mle_short = fit_gev(short_record, method="lmom")
params_map_short = fit_gev_map(short_record)

print("\nShort record (15 years) — return levels at T=100:")
print(f"  MLE: {return_level_gev(params_mle_short, 100):.0f} m³/s  "
      f"xi = {params_mle_short['xi']:.3f}")
print(f"  MAP: {return_level_gev(params_map_short, 100):.0f} m³/s  "
      f"xi = {params_map_short['xi']:.3f}  (prior pulls xi toward 0)")


Short record (15 years) — return levels at T=100:
  MLE: 4343 m³/s  xi = 0.241
  MAP: 4290 m³/s  xi = 0.177  (prior pulls xi toward 0)


### 5.2 Full MCMC posterior (PyMC + NUTS)

Full MCMC samples the **entire posterior distribution** p(μ, σ, ξ | data), giving:
- A **distribution** over return levels, not just a point estimate
- Honest uncertainty intervals that widen for larger return periods (more extrapolation)
- Detection of **bimodal posteriors** (multiple plausible parameter sets) invisible to MLE

**NUTS sampler (No U-Turn Sampler):** Efficient Hamiltonian Monte Carlo that auto-tunes step size. Much faster than random-walk Metropolis for continuous parameters.

**Non-centered parameterisation:** Internal reparameterisation μ = ȳ + σ_y × μ_raw where μ_raw ~ N(0,1) improves NUTS mixing (reduces posterior correlation between μ and σ).

```bash
pip install pymc
```

> **Production settings:** n_chains=4, n_samples=2000. Check R̂ < 1.01 and ESS > 400 in the posterior summary.

In [19]:
# Full MCMC posterior via PyMC + NUTS (non-centered parameterisation)
# Uses 2 chains / 500 samples for speed; use 4 chains / 2000 for production.
posterior = fit_gev_mcmc(annual_max.values, n_samples=500, n_chains=2, adapt_delta=0.90)
print("MCMC posterior summary (mu, sigma, xi):")
print(posterior.describe().round(1))


MCMC posterior summary (mu, sigma, xi):
           mu   sigma      xi
count  1000.0  1000.0  1000.0
mean    873.8   468.2     0.1
std      68.2    56.1     0.1
min     636.5   330.5    -0.1
25%     827.2   429.1     0.1
50%     874.4   464.9     0.1
75%     920.2   500.9     0.2
max    1056.4   763.1     0.6


/usr/local/lib/python3.11/site-packages/arviz/__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(
Initializing NUTS using jitter+adapt_diag...
/usr/local/lib/python3.11/site-packages/pytensor/link/c/cmodule.py:2986: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pyte

In [20]:
# Posterior distribution of the T=100 return level.
# Each MCMC sample is a plausible parameter set → a distribution over return levels.
T_post = 100
rl_posterior = np.array([
    return_level_gev({'mu': r['mu'], 'sigma': r['sigma'], 'xi': r['xi']}, T_post)
    for _, r in posterior.iterrows()
])
# Remove extreme tails (xi > 0.5 can produce very large extrapolations)
rl_posterior = rl_posterior[np.isfinite(rl_posterior) & (rl_posterior < 1e6)]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(rl_posterior, bins=50, density=True, color="steelblue", alpha=0.7)
ax.axvline(np.median(rl_posterior), color="k", ls="--", lw=1.5,
           label=f"Median: {np.median(rl_posterior):.0f} m³/s")
ax.axvline(np.quantile(rl_posterior, 0.05), color="r", ls=":", lw=1.2,
           label=f"5th pct: {np.quantile(rl_posterior, 0.05):.0f} m³/s")
ax.axvline(np.quantile(rl_posterior, 0.95), color="r", ls=":", lw=1.2,
           label=f"95th pct: {np.quantile(rl_posterior, 0.95):.0f} m³/s")
ax.set_xlabel(f"T={T_post} return level (m³/s)")
ax.set_ylabel("Posterior density")
ax.set_title(f"MCMC posterior of T={T_post} return level", fontsize=11)
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nT={T_post} return level:")
print(f"  Median : {np.median(rl_posterior):.0f} m³/s")
print(f"  90% CI : [{np.quantile(rl_posterior, 0.05):.0f}, {np.quantile(rl_posterior, 0.95):.0f}] m³/s")



T=100 return level:
  Median : 3844 m³/s
  90% CI : [2971, 5841] m³/s


### Interpreting the MCMC posterior

The histogram above shows the **posterior predictive distribution of the T=100 return level** — each bar represents a plausible return level value given the data and priors.

Key quantities to report:
- **Median**: robust central estimate, less sensitive to heavy-tail samples than the mean
- **90% credible interval [5th, 95th percentile]**: probability 0.90 that the true T=100 return level lies within this range *given the model and data*
- **Width of the interval**: directly reflects record length — longer records → narrower intervals

> **Important distinction:** A credible interval (Bayesian) is *not* the same as a confidence interval (frequentist). The credible interval directly gives the probability that the parameter lies in the range. The confidence interval is a property of the estimation procedure.

---
## 6. Seasonal analysis

Extract flood peaks per season and fit a GPD to each — useful when flood
generating mechanisms differ between seasons (e.g. snowmelt in spring vs
convective storms in summer).

**Seasonal POT approach** (preferred over seasonal block maxima):
* Seasonal block maxima include many "no-flood" years as base-flow values,
  producing highly skewed samples that break GEV MLE.
* Filtering POT peaks by season avoids this: only actual flood events enter
  the fit.
* `lambda_rate` is expressed as peaks **per calendar year** (not per season),
  so return periods T remain in years.

In [21]:
season_map = {12: "DJF", 1: "DJF", 2: "DJF",
              3: "MAM", 4: "MAM", 5: "MAM",
              6: "JJA", 7: "JJA", 8: "JJA",
              9: "SON", 10: "SON", 11: "SON"}

n_cal_years = discharge.index.year.nunique()   # 60 calendar years
T_ref = 100

fig, axes = plt.subplots(2, 2, figsize=(12, 7))

for ax, season in zip(axes.ravel(), ["DJF", "MAM", "JJA", "SON"]):
    # Filter discharge to this season's months
    seas_discharge = discharge[discharge.index.month.map(season_map) == season]

    # POT peaks within this season only
    seas_peaks = extract_pot(seas_discharge, threshold=params_gpd["threshold"], min_gap=7)

    if len(seas_peaks) < 10:
        ax.set_title(f"{season} — insufficient peaks ({len(seas_peaks)})")
        continue

    # Fit GPD to seasonal exceedances
    try:
        params_seas = fit_gpd(
            seas_discharge,
            threshold=params_gpd["threshold"],
            method="mle",
            min_gap=7,
        )
        # Override lambda with peaks-per-*calendar*-year so T is in years
        params_seas["lambda_rate"] = len(seas_peaks) / n_cal_years

        rl = return_level_gpd(params_seas, T_ref)
        plot_return_levels(seas_discharge, params_seas, dist="gpd", n_bootstrap=50, ax=ax)
        ax.set_title(
            f"{season}  n={len(seas_peaks)}  T={T_ref}: {rl:.0f} m³/s  "
            f"ξ={params_seas['shape']:.3f}",
            fontsize=9,
        )
    except Exception as e:
        ax.set_title(f"{season} — fit failed: {e}", fontsize=9)

plt.suptitle("Seasonal GPD (POT) — return level plots", fontsize=12)
plt.tight_layout()
plt.show()

---
## 7. Non-stationarity check

A simple trend check on annual maxima — if the Mann-Kendall test is
significant, a non-stationary GEV (trend in location parameter) may be
more appropriate.

In [22]:
from scipy.stats import kendalltau, linregress

years = annual_max.index.year.values
values_am = annual_max.values

tau, p_mk = kendalltau(years, values_am)
slope, intercept, r, p_lr, _ = linregress(years, values_am)

print(f"Mann-Kendall: tau = {tau:.3f}  p = {p_mk:.3f}")
print(f"Linear trend: slope = {slope:.2f} m³/s/year  p = {p_lr:.3f}")
if p_mk < 0.05:
    print("→ Statistically significant trend detected — consider non-stationary analysis.")
else:
    print("→ No significant trend — stationary GEV/GPD is appropriate.")

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(years, values_am, s=25, color="steelblue", zorder=4)
ax.plot(years, slope * years + intercept, "r--", lw=1.5,
        label=f"Trend: {slope:.1f} m³/s/yr (p={p_lr:.3f})")
ax.set_xlabel("Year")
ax.set_ylabel("Annual maximum (m³/s)")
ax.set_title("Annual maxima — trend analysis", fontsize=11)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Mann-Kendall: tau = 0.028  p = 0.750
Linear trend: slope = 0.26 m³/s/year  p = 0.962
→ No significant trend — stationary GEV/GPD is appropriate.
